# NLP 2026
# Lab 3: Classification and BERT

Have you ever read a movie review and wondered:

```“Is this review actually positive or negative?” 🤔```

In this lab, you will build your own sentiment analysis tool using Natural Language Processing (NLP)! Your goal is to automatically classify movie reviews into one of two categories:

✅ Positive

❌ Negative

We will approach this as a binary classification task and you will experiment with increasingly powerful methods — from classic machine learning to modern neural networks based on transformers 🚀

### 🎯 Learning Goals ###

By completing this lab, you should be able to:

- Formulate sentiment analysis as a binary classification problem

- Design and evaluate hand-crafted text features

- Implement a Bag-of-Words representation

- Apply and evaluate Logistic Regression and alternative classifiers

- Understand how BERT tokenization and embeddings work

- Extract sentence representations using: ```CLS``` token, mean token pooling

- Compare classical ML and transformer-based methods

- Critically analyze evaluation metrics beyond accuracy 📊



### Score breakdown

| Exercise            | Points |
|---------------------|--------|
| [Exercise 1](#e1)   | 1      |
| [Exercise 2](#e2)   | 5      |
| [Exercise 3](#e3)   | 5      |
| [Exercise 4](#e4)   | 5      |
| [Exercise 5](#e5)   | 5      |
| [Exercise 6](#e6)   | 5      |
| [Exercise 7](#e7)   | 3      |
| [Exercise 8](#e8)   | 6      |
| [Exercise 9](#e9)   | 5      |
| [Exercise 10](#e10) | 5      |
| [Exercise 12](#e12) | 5      |
| [Exercise 13](#e13) | 10     |
| Total               | 60     |

This score will be scaled down to 0.6 and that will be your final lab score.

### 📌 **Instructions for Delivery** (📅 **Deadline: 23/Feb 18:00**, 🎭 *wildcards possible*)

✅ **Submission Requirements**
+ 📄 You need to submit a **notebook** 📓 with the code, appropriate comments and figures in all questions. Make sure to have a mix of code (some explanations needed if not clear what you implement), figures to support the answers or your claims and proper amount of text to explain your reasoning, answer etc.
+ ⚡ Make sure that **all cells are executed properly** ⚙️ and that **all figures/results/plots** 📊 you include in the report are also visible in your **executed notebook**.
+ You can work on Google Collab (or other environments), but you need to make sure that your delivered notebook is executed properly.

✅ **Collaboration & Integrity**
+ 🗣️ While you may **discuss** the lab with others, you must **write your solutions with your group only**. If you **discuss specific tasks** with others, please **include their names** below.
+ 📜 **Honor Code applies** to this lab. For more details, check **Syllabus §7.2** ⚖️.
+ 📢 **Mandatory Disclosure**:
   - Any **websites** 🌐 (e.g., **Stack Overflow** 💡) or **other resources** used must be **listed and disclosed**.
   - Any **GenAI tools** 🤖 (e.g., **ChatGPT**) used must be **explicitly mentioned**.
   - 🚨 **Failure to disclose these resources is a violation of academic integrity**. See **Syllabus §7.3** for details.

## 0. Setup

We first install the scikit-learn library [Scikit-learn](https://scikit-learn.org/stable/). We will use its classification models.

In [1]:
pip install -U scikit-learn

Note: you may need to restart the kernel to use updated packages.


We will need [PyTorch](https://pytorch.org/) installed. It is a very popular deep learning library that offers modularized versions of many of the sequence models we discussed in class. It's an important tool that you may want to practice further if you want to dive deeper into NLP, since much of the current academic and industrial research uses it.

Some resources to look further are given below.

* [Documentation](https://pytorch.org/docs/stable/index.html) (We will need this soon)

* [Installation Instructions](https://pytorch.org/get-started/locally/)

* [Quickstart Tutorial](https://pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html)

The cell below should install the library:

In [2]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


The last bit we need is the huggingface transformers library (here is the documentation [https://huggingface.co/docs/transformers/en/index](https://huggingface.co/docs/transformers/en/index)). Transformers are one of the most influential architectures in handling sequences (not only in language). As we discussed in lectures, they excel at taking into account context (which is the salt-and-pepper of NLP) with mechansisms such as self-attetion, which allows them to weigh the importance of different words in a sentence. If you want to know more, revisit the course material (slides and textbook).

We already used huggingface datasets in previous labs and huggingface transformers integrates nicely with that. Apart from the ease of use, huggingface is also providing pre-trained models of different kinds. The list can be found [here](https://huggingface.co/models) ([https://huggingface.co/models](https://huggingface.co/models)). The following line should be enough to install huggingface transformers library:

In [3]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


Here, we import the libraries:

In [1]:
import re
from collections import Counter
#!pip install datasets
import datasets
import numpy as np
import torch
import tqdm
import transformers
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from torch.utils.data.dataloader import DataLoader

## 1. Loading the Dataset

We will work with the IMDB dataset [https://huggingface.co/datasets/stanfordnlp/imdb](https://huggingface.co/datasets/stanfordnlp/imdb). It contains the reviews and a label that indicates whether the review is positive or not (the neutral reviews have been filtered out). You can read the paper [here](https://aclanthology.org/P11-1015/).

In [2]:
dataset = datasets.load_dataset('stanfordnlp/imdb', split=['train', 'test'])
print(dataset)

[Dataset({
    features: ['text', 'label'],
    num_rows: 25000
}), Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})]


Notice that the dataset has been loaded as a list of two datasets. They are the `train` and `test` splits that we asked for.
We will use the validation subset to tune the parameters. So, let's split the `train` subset and create a `DatasetDict` object:

In [3]:
train_valid_split = dataset[0].train_test_split(5000)
dataset = datasets.DatasetDict({
    'train': train_valid_split['train'].shuffle(),
    'validation': train_valid_split['test'].shuffle(),
    'test': dataset[1].shuffle(),
})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})


We can print several examples from the `train` dataset:

In [4]:
for i in range(5):
    print('i', i)
    print(dataset['train'][i]['text'])
    print(dataset['train'][i]['label'])
    print()

i 0
I suppose that today this film has relevance because it was an early Sofia Loren film. She was 19 years old when the film was made in 1953.<br /><br />I viewed this film because I wanted to see some of Sofia Loren's early work. I was surprised when she came on camera having had her skin bronzed over in brown makeup to resemble an Ethiopian princess. Surely, today, this would have been viewed as a slur and to be avoided in movie making. It actually became annoying watching Ms. Loren in skin color paint throughout the film.<br /><br />Yes, this film would have been better made if the real opera singers had made this movie. Then, the singing and the actual facial gestures of the real artists would have been apparent. I discount the comments by others about whether the real opera singers are older and heavier in weight.<br /><br />As beautiful as Ms. Loren was at age 19 and still is today, the film would have been better received as though it were being performed on the stage. After al

Let's extract the labels from the dataset. We will use them to train and evaluate our classifiers.

In [5]:
y_train = dataset['train']['label']
print(y_train)
y_valid = dataset['validation']['label']
print(y_valid)

Column([0, 1, 0, 0, 0, ...])
Column([1, 1, 1, 0, 0, ...])


<a name='e1'></a>
#### Exercise 1: Cleaning the text

(1p) In this exercise you should clean the text in the dataset. This is the same step we saw in the previous labs.

If you think this step is not necessary in this use case, you can skip this step, but make sure to justify your decision.

In [6]:
def clean(text):
    """
    Cleans the text
    Args:
        text: a string that will be cleaned

    Returns: the cleaned text

    """

    # Empty text
    if text == '':
        return text

   # Lowercase
    text = text.lower()

    # Remove common HTML break
    text = text.replace("<br />", " ")

    # Remove punctuation manually
    for p in ".,!?;:()[]{}\"'":
        text = text.replace(p, "")

    # Remove extra spaces
    text = " ".join(text.split())


    return text


def clean_example(example):
    """
    Applies the clean() function to the example from the Dataset
    Args:
        example: an example from the Dataset

    Returns: update example with cleaned 'text' column

    """
    example['text'] = clean(example['text'])
    return example


dataset = dataset.map(clean_example, desc="Cleaning")
print(dataset)

Cleaning:   0%|          | 0/20000 [00:00<?, ? examples/s]

Cleaning:   0%|          | 0/5000 [00:00<?, ? examples/s]

Cleaning:   0%|          | 0/25000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})


## 2. Hand-crafted Features

<a name='e2'></a>
#### Exercise 2: Hand-crafted features

(5p) Write your own hand-crafted feature extraction function. Include at least these types of features:
- length of the text,
- number of different punctuation characters,
- number of positive and negative words.

In [7]:
### YOUR CODE HERE
# you can define the positive and negative words here

positive_words = ["good", "great", "excellent", "amazing", "love", "wonderful", "best", "fantastic"]
negative_words = ["bad", "terrible", "awful", "boring", "worst", "hate", "poor", "disappointing"]



def calculate_features(text):
    features = []
    ### YOUR CODE HERE
    # Length of text 
    words = text.split()
    length = len(words)
    features.append(length)
    
    # Number of different punctuation characters
    punctuation_chars = ".,!?;:()[]{}\"'"
    different_punct = len(set([c for c in text if c in punctuation_chars]))
    features.append(different_punct)

    # Number of positive words
    pos_count = sum(1 for w in words if w in positive_words)
    features.append(pos_count)

    # Number of negative words
    neg_count = sum(1 for w in words if w in negative_words)
    features.append(neg_count)



    ### YOUR CODE ENDS HERE
    return np.array(features, dtype=float)


### YOUR CODE ENDS HERE

The function below will apply your feature extraction implementation to a specified dataset.

In [8]:
def calculate_features_dataset(dataset, features_fn):
    all_features = []
    for e in tqdm.tqdm(dataset, desc='Extracting features'):
        text = e['text']
        features = features_fn(text)
        all_features.append(features)
    all_features = np.array(all_features, dtype=float)
    return all_features

And we can obtain the features for the `train` and `validation` splits. Later you will need to do the same for the `test` subset.

In [9]:
X_train = calculate_features_dataset(dataset['train'], calculate_features)
X_valid = calculate_features_dataset(dataset['validation'], calculate_features)

Extracting features: 100%|███████████████████████████████████| 5000/5000 [00:00<00:00, 16025.32it/s]


### 2.1 Classification

In this section, we will create and train a logistic regression classifier. We will train it on the `train` subset and evaluate on the `validation` split. Later, you will do a final comparison between methods on the `test` subset, but it is important to avoid it when tuning the methods.

In [10]:
classifier = LogisticRegression(solver='lbfgs', max_iter=1000)
classifier.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

Let's check the performance on the training data:

In [11]:
print('Features results: train')
pred_train = classifier.predict(X_train)
print(accuracy_score(y_train, pred_train))

Features results: train
0.7133


... and the validation dataset:

In [12]:
print('Features results: validation')
pred_valid = classifier.predict(X_valid)
print(accuracy_score(y_valid, pred_valid))

Features results: validation
0.7086


<a name='e3'></a>
#### Exercise 3: Improving the features

(5p) Iteratively improve your hand-crafted features. Think about what information from the review might be useful for to predict the rating a person gave to the particular movie. You can also look into the expected format (or range) of features for the classifier.

Document the steps you tried (even if unsuccessful) and how they influenced the metrics. Try at least 3 modifications from your original implementation.

In [13]:
### YOUR CODE HERE
def calculate_features_v2(text):
    features = []

    words = text.split()
    n_words = len(words)
    n_chars = len(text)

    # length words
    features.append(n_words)

    # different punctuation characters
    punctuation_chars = ".,!?;:()[]{}\"'"
    different_punct = len(set([c for c in text if c in punctuation_chars]))
    features.append(different_punct)

    # helper for counting words
    def count_in_list(words, vocab):
        return sum(1 for w in words if w in vocab)

    pos_count = count_in_list(words, positive_words)
    neg_count = count_in_list(words, negative_words)

    #  pos/neg counts
    features.append(pos_count)
    features.append(neg_count)

    # Improvements

    #  Pos-Neg difference (sentiment balance)
    features.append(pos_count - neg_count)

    #  Pos/Neg ratios (normalize by length)
    # avoid division by 0
    denom = n_words if n_words > 0 else 1
    features.append(pos_count / denom)
    features.append(neg_count / denom)

    # Exclamation and question marks counts (intensity / rhetorical)
    features.append(text.count("!"))
    features.append(text.count("?"))

    # ALL CAPS words count 
    caps_count = sum(1 for w in text.split() if w.isupper() and len(w) > 1)
    features.append(caps_count)

    # Average word length 
    avg_word_len = (sum(len(w) for w in words) / denom) if denom > 0 else 0.0
    features.append(avg_word_len)

    return np.array(features, dtype=float)


    ### YOUR CODE ENDS HERE
X_train_v2 = calculate_features_dataset(dataset['train'], calculate_features_v2)
X_valid_v2 = calculate_features_dataset(dataset['validation'], calculate_features_v2)

classifier_v2 = LogisticRegression(solver="lbfgs", max_iter=1000)
classifier_v2.fit(X_train_v2, y_train)

print("Improved features results: train")
pred_train_v2 = classifier_v2.predict(X_train_v2)
print(accuracy_score(y_train, pred_train_v2))

print("Improved features results: validation")
pred_valid_v2 = classifier_v2.predict(X_valid_v2)
print(accuracy_score(y_valid, pred_valid_v2))

Extracting features: 100%|███████████████████████████████████| 5000/5000 [00:00<00:00, 13492.30it/s]


Improved features results: train
0.71435
Improved features results: validation
0.7194


--- YOUR ANSWERS HERE
Baseline (Exercise 2):
Train accuracy ≈ 0.7139;   Validation accuracy ≈ 0.7102

After improvements (Exercise 3):
Train accuracy ≈ 0.71395;   Validation accuracy ≈ 0.7168

Modifications tried:
- Sentiment balance feature: added (pos_count - neg_count). Helps capture whether the review is overall more positive than negative, not just raw counts.
- Length normalization: added pos_count/len(words) and neg_count/len(words). Makes counts comparable across short vs long reviews, reduces bias toward long texts.
- Intensity/style: added ! count, ? count, ALL-CAPS count, and average word length. Captures emphasis and writing style; the effect was small but slightly improved validation.

Conclusion:
The reason why the improvement is modest is that these features are quite coarse and the IMDB sentiment is driven by a lot of specific words/phrases. Usually, the performance improvement is driven by more complex features such as Bag-of-Words/TF-IDF or pre-trained embeddings.

<a name='e4'></a>
#### Exercise 4: Improving the evaluation

(5p) In the previous cells, we only looked at the accuracy of predictions. Investigate which other metrics might be better for our case. You can check the documentation of scikit-learn for evaluation metrics ([https://scikit-learn.org/stable/api/sklearn.metrics.html#classification-metrics](https://scikit-learn.org/stable/api/sklearn.metrics.html#classification-metrics)). Give reasons why the metrics you try can be more informative than raw accuracy score.

Decide which evaluation metric(s) is most suitable for our use-case and give reasons why. Test your features-based classifier and all further classifiers on that metric (apart from the accuracy score).

In [14]:
### YOUR CODE HERE
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

pred_valid = pred_valid_v2   

print("Accuracy:", accuracy_score(y_valid, pred_valid))
print("Precision:", precision_score(y_valid, pred_valid))
print("Recall:", recall_score(y_valid, pred_valid))
print("F1:", f1_score(y_valid, pred_valid))

print("\nConfusion matrix\n [ [TN FP] [FN TP] ]:")
print(confusion_matrix(y_valid, pred_valid))

print("\nClassification report:")
print(classification_report(y_valid, pred_valid, digits=4))

### YOUR CODE ENDS HERE

Accuracy: 0.7194
Precision: 0.691493116837275
Recall: 0.787379421221865
F1: 0.7363277579402367

Confusion matrix
 [ [TN FP] [FN TP] ]:
[[1638  874]
 [ 529 1959]]

Classification report:
              precision    recall  f1-score   support

           0     0.7559    0.6521    0.7001      2512
           1     0.6915    0.7874    0.7363      2488

    accuracy                         0.7194      5000
   macro avg     0.7237    0.7197    0.7182      5000
weighted avg     0.7238    0.7194    0.7182      5000



--- YOUR ANSWERS HERE
Accuracy alone does not fully describe the classifier’s behavior because it does not distinguish between different types of errors. Precision and recall provide more detailed insight. Precision measures how many predicted positive reviews are truly positive, while recall measures how many truly positive reviews are correctly identified. In our results, recall (0.783) is higher than precision (0.688), meaning the model tends to predict the positive class more often and produces more false positives than false negatives.
The F1-score (0.733) balances precision and recall and is therefore more informative than accuracy for evaluating classification performance. The confusion matrix confirms this behavior, showing a relatively high number of false positives compared to false negatives.
For this task, the F1-score is a suitable main metric because it considers both precision and recall and provides a balanced evaluation of model performance.

## 3. Bag-of-Words Classifier

Similar to the previous lab, we will use the classic bag-of-words representation as one of our embeddings. While it is simple and does not preserve the positions of words, it gives our classifier a lot of useful information.

<a name='e5'></a>
#### Exercise 5: Implementing BOW

(5p) Implement the BOW. In this exercise, we do not give you a rigid structure, so you can conjure your own. The two things your code should produce is the `token_to_id` dictionary, and `bag_of_words()` function that accepts a list of tokens, and the `token_to_id` dictionary while generating the BOW representation as a numpy array.

In [15]:
#### YOUR CODE HERE

MAX_VOCAB_SIZE = 1_000

# The goal is to implement the `bag_of_words(tokens, token_to_id)` function similar to the previous lab.
# You might want to follow the steps:
# - tokenize the `text` column in the dataset,
# - extract the vocabulary from the tokens,
# - limit the vocabulary to `MAX_VOCAB_SIZE`,
# - calculate the `token_to_id` dictionary
# - implement the `bag_of_words(tokens, token_to_id)` function.
all_tokens = []

for example in dataset['train']:
    tokens = example['text'].split()
    all_tokens.extend(tokens)

# Count word frequencies
counter = Counter(all_tokens)

# Keep most frequent words
most_common = counter.most_common(MAX_VOCAB_SIZE)

token_to_id = {word: idx for idx, (word, _) in enumerate(most_common)}
print("Vocabulary size:", len(token_to_id))

def add_tokens(example):
    example["tokens"] = example["text"].split()
    return example

dataset = dataset.map(add_tokens)

def bag_of_words(tokens, token_to_id):
    """
    Creates a bag-of-words representation of the sentence
    Args:
        tokens: a list of tokens
        token_to_id: a dictionary mapping each word to an index in the vocabulary

    Returns:: a numpy array of size vocab_size with the counts of each word in the vocabulary
    """
    vocab_size = len(token_to_id)
    bow = np.zeros(vocab_size)

    for token in tokens:
        if token in token_to_id:
            idx = token_to_id[token]
            bow[idx] += 1
    return bow 




#### YOUR CODE ENDS HERE

Vocabulary size: 1000


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Here, we will use your implemented function to calculate the bag-of-words for each example in the `train` and `validation` subsets.

In [16]:
train_bows = []
for example in tqdm.tqdm(dataset['train'], desc='Calculating test BOWs'):
    train_bows.append(bag_of_words(example['tokens'], token_to_id))
train_bows = np.array(train_bows, dtype=float)

valid_bows = []
for example in tqdm.tqdm(dataset['validation'], desc='Calculating validation BOWs'):
    valid_bows.append(bag_of_words(example['tokens'], token_to_id))
valid_bows = np.array(valid_bows, dtype=float)

Calculating validation BOWs: 100%|████████████████████████████| 5000/5000 [00:00<00:00, 8235.63it/s]


Finally, we can train the classifier on the BOW representations and the labels in the `train` split.

In [17]:
classifier = LogisticRegression(solver='lbfgs', max_iter=1000)
print('Training classifier...')
classifier.fit(train_bows, y_train)

Training classifier...


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

Let's evaluate the classifier:

In [18]:
print('BOW results: train')
pred_train = classifier.predict(train_bows)
print(accuracy_score(y_train, pred_train))

print('BOW results: validation')
pred_valid = classifier.predict(valid_bows)
print(accuracy_score(y_valid, pred_valid))

BOW results: train
0.87925
BOW results: validation
0.8552


<a name='e6'></a>
#### Exercise 6: Tuning the model

(5p) Try different values for the vocab size. Experiment with adding the hand-crafted features. Test the model on the evaluation metric of your choice (remember to use the validation split).

In [19]:
#### YOUR CODE HERE
VOCAB_SIZES = [500, 1000, 2000, 5000, 10000]

def build_token_to_id(dataset_split, max_vocab_size):
    all_tokens = []
    for ex in dataset_split:
        all_tokens.extend(ex["tokens"])
    counter = Counter(all_tokens)
    most_common = counter.most_common(max_vocab_size)
    return {w: i for i, (w, _) in enumerate(most_common)}

def build_bow_matrix(dataset_split, token_to_id):
    bows = []
    for ex in tqdm.tqdm(dataset_split, desc=f"BoW (V={len(token_to_id)})"):
        bows.append(bag_of_words(ex["tokens"], token_to_id))
    return np.array(bows, dtype=float)

results = []

for V in VOCAB_SIZES:
    print("\n" + "-"*60)
    print(f"Vocab size = {V}")

    # 1) vocab
    token_to_id_v = build_token_to_id(dataset["train"], V)

    # 2) BoW matrices
    X_train_bow = build_bow_matrix(dataset["train"], token_to_id_v)
    X_valid_bow = build_bow_matrix(dataset["validation"], token_to_id_v)

    # 3) BoW-only model
    clf = LogisticRegression(solver="lbfgs", max_iter=1000)
    clf.fit(X_train_bow, y_train)
    pred_valid = clf.predict(X_valid_bow)

    acc = accuracy_score(y_valid, pred_valid)
    f1 = f1_score(y_valid, pred_valid)
    print(f"BoW only   -> acc={acc:.4f}, f1={f1:.4f}")

    # 4) BoW + handcrafted features
    X_train_feat = calculate_features_dataset(dataset["train"], calculate_features_v2)
    X_valid_feat = calculate_features_dataset(dataset["validation"], calculate_features_v2)

    X_train_combo = np.concatenate([X_train_bow, X_train_feat], axis=1)
    X_valid_combo = np.concatenate([X_valid_bow, X_valid_feat], axis=1)

    clf2 = LogisticRegression(solver="lbfgs", max_iter=1000)
    clf2.fit(X_train_combo, y_train)
    pred_valid2 = clf2.predict(X_valid_combo)

    acc2 = accuracy_score(y_valid, pred_valid2)
    f12 = f1_score(y_valid, pred_valid2)
    print(f"BoW+feat   -> acc={acc2:.4f}, f1={f12:.4f}")

    results.append((V, acc, f1, acc2, f12))

print("\nSummary (V, acc_bow, f1_bow, acc_combo, f1_combo):")
for row in results:
    print(row)

#### YOUR CODE ENDS HERE


------------------------------------------------------------
Vocab size = 500


BoW (V=500): 100%|████████████████████████████████████████████| 5000/5000 [00:00<00:00, 8233.73it/s]


BoW only   -> acc=0.8400, f1=0.8397


Extracting features: 100%|████████████████████████████████████| 5000/5000 [00:00<00:00, 6401.52it/s]
/opt/anaconda3/envs/nlplab2026/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


BoW+feat   -> acc=0.8388, f1=0.8380

------------------------------------------------------------
Vocab size = 1000


...

BoW+feat   -> acc=0.8718, f1=0.8712

------------------------------------------------------------
Vocab size = 10000


BoW (V=10000): 100%|██████████████████████████████████████████| 5000/5000 [00:00<00:00, 7108.00it/s]


BoW only   -> acc=0.8826, f1=0.8824


Extracting features: 100%|████████████████████████████████████| 5000/5000 [00:00<00:00, 6360.17it/s]
/opt/anaconda3/envs/nlplab2026/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


BoW+feat   -> acc=0.8810, f1=0.8809

Summary (V, acc_bow, f1_bow, acc_combo, f1_combo):
(500, 0.84, 0.8397435897435898, 0.8388, 0.8380225080385852)
(1000, 0.8552, 0.8557194101235552, 0.8566, 0.8569718731298623)
(2000, 0.865, 0.8645394340758579, 0.8666, 0.8658217662442165)
(5000, 0.8688, 0.8689048760991207, 0.8718, 0.8712075547518585)
(10000, 0.8826, 0.8823882989380886, 0.881, 0.8808808808808809)


## 4. BERT Model

For the first part of this lab, we will be using a pre-trained BERT model from Huggingface, namely the [BERT Cased](https://huggingface.co/google-bert/bert-base-cased). You can read the original paper that introduced this model [here](https://aclanthology.org/N19-1423.pdf). This paper has been one of the most cited papers ever (currently having more than 100,000 citations).

We will specify the model name that can be found on the model's card on huggingface (revisit the first link). Make sure to check what other information Huggingface is offering (e.g. how to use the model, limitations, how to inference, etc.).

In [20]:
model_name = 'google-bert/bert-base-cased'

### 4.1 Tokenizer

The models on huggingface come with their own tokenizers. They are loaded separately from the models. We can use [AutoTokenizer](https://huggingface.co/docs/transformers/v4.40.2/en/model_doc/auto#transformers.AutoTokenizer)'s `from_pretrained()` method to load it.

Inspect the output: The loaded object is of `BertTokenizer` class. Check the documentation [here](https://huggingface.co/docs/transformers/en/model_doc/bert#transformers.BertTokenizer).

In [21]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
print(tokenizer)

BertTokenizer(name_or_path='google-bert/bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


Next, let's see how we can use it to tokenize some text.

In [22]:
print(dataset['test'][0]['text'])
tokenized = tokenizer(dataset['test'][0]['text'], padding=True, return_tensors='pt')
print("---")
print(type(tokenized))
print("---")
print(tokenized)

this brief review contains no spoilers since the movie spoils itself it is wooden and pedantic it has no saving grace whatsoever if someone invites you to his house to watch mr imperium dont go even the title of the movie is dreadful and portends what garbage lies within the whole plot is so bad that it could drive mother theresa to despair it wasnt a stroke that led to the early demise of poor ezio it was having to act in this clunker that did him in it must have haunted him the rest of his days perhaps he was an enemy alien and wanted revenge upon the americans for his confinement he found a perfect vehicle for his wrath in this travesty
---
<class 'transformers.tokenization_utils_base.BatchEncoding'>
---
{'input_ids': tensor([[  101,  1142,  4094,  3189,  2515,  1185,   188,  5674, 25614,  1116,
          1290,  1103,  2523,   188,  5674,  8825,  2111,  1122,  1110,  4122,
          1105,   185, 14604, 14964,  1665,  1122,  1144,  1185,  7740, 11116,
         20748,  1191,  1800, 20

Examine the outputs: The tokenizer returned three things:
- `input_ids` - this is a PyTorch tensor ([https://pytorch.org/docs/stable/tensors.html](https://pytorch.org/docs/stable/tensors.html)) with the indices of our tokens. PyTorch tensors are similar to numpy arrays. They hold data in a multidimensional array or matrix. The difference is that PyTorch tensors can be placed and modified on the GPU which greatly improves the speed of execution.
- `token_type_ids` - this tensor holds the information about the index of the sentence. This has to do with the classification objective from the original paper, where two sentences were given and the model had to predict if they are connected. Because we only included a single sentence, we have only zeros here. We will not be concerned with it in this lab.
- `attention_mask` - holds the mask that the model will use to determine if the tokens in the `input_ids` are the real tokens or *padding*. Padding is a technique used to ensure that all input sequences have the same length. BERT (like many other NLP models) process data in batches and requires each sequence in a batch to have the same length, so sequences that are shorter than the maximum sequence length in the batch are padded with special tokens. In this case, because we only inputted a single sentence, the mask contains only ones. Later you will see examples where this is not the case.

Let's see how exactly the sentence was tokenized and how we can retrieve the original text. Notice that some words have been split into multiple tokens (remember when we discussed sub-word tokenization in class?). Also pay attention to the added special tokens, namely `CLS` and `SEP`:

The `[CLS]` token is a special classification token added at the beginning of every input sequence. It stands for "classification" (daah!) and is used by BERT to aggregate information from the entire sequence. The final hidden state corresponding to this token (after passing through the transformer layers) is used as the aggregate sequence representation for classification tasks. We will use this later in the lab!

The `[SEP]` token is used to separate different segments or sentences within the input sequence. It stands for "separator" (daaah again!).

In [23]:
print(tokenized['input_ids'].shape)
print("---")
print(tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0]))
print("---")
print(len(tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0])))
print("---")
print(tokenizer.decode(tokenized['input_ids'][0]))
print("---")
print(tokenizer.decode(tokenized['input_ids'][0], skip_special_tokens=True))

torch.Size([1, 149])
---
['[CLS]', 'this', 'brief', 'review', 'contains', 'no', 's', '##po', '##iler', '##s', 'since', 'the', 'movie', 's', '##po', '##ils', 'itself', 'it', 'is', 'wooden', 'and', 'p', '##eda', '##nti', '##c', 'it', 'has', 'no', 'saving', 'grace', 'whatsoever', 'if', 'someone', 'invites', 'you', 'to', 'his', 'house', 'to', 'watch', 'm', '##r', 'imp', '##eri', '##um', 'don', '##t', 'go', 'even', 'the', 'title', 'of', 'the', 'movie', 'is', 'dreadful', 'and', 'port', '##ends', 'what', 'garbage', 'lies', 'within', 'the', 'whole', 'plot', 'is', 'so', 'bad', 'that', 'it', 'could', 'drive', 'mother', 'there', '##sa', 'to', 'despair', 'it', 'wasn', '##t', 'a', 'stroke', 'that', 'led', 'to', 'the', 'early', 'demise', 'of', 'poor', 'e', '##zio', 'it', 'was', 'having', 'to', 'act', 'in', 'this', 'c', '##lu', '##nk', '##er', 'that', 'did', 'him', 'in', 'it', 'must', 'have', 'haunted', 'him', 'the', 'rest', 'of', 'his', 'days', 'perhaps', 'he', 'was', 'an', 'enemy', 'alien', 'and', 

Tokenizer can process a list of sentences. This will create a batched output with tensor's first dimension corresponding to the batch size (the number of sentences we passed to the tokenizer). Examine the following cell and make sure it makes sense to you.

In [24]:
print(dataset['test'][0:3]['text'])
tokenized = tokenizer(dataset['test'][0:3]['text'], padding=True, return_tensors='pt')
print(tokenized)
print(tokenized['input_ids'].shape)
print(tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0]))
print(len(tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0])))
print(tokenizer.decode(tokenized['input_ids'][0]))
print(tokenizer.decode(tokenized['input_ids'][0], skip_special_tokens=True))

['this brief review contains no spoilers since the movie spoils itself it is wooden and pedantic it has no saving grace whatsoever if someone invites you to his house to watch mr imperium dont go even the title of the movie is dreadful and portends what garbage lies within the whole plot is so bad that it could drive mother theresa to despair it wasnt a stroke that led to the early demise of poor ezio it was having to act in this clunker that did him in it must have haunted him the rest of his days perhaps he was an enemy alien and wanted revenge upon the americans for his confinement he found a perfect vehicle for his wrath in this travesty', 'i know little or nothing about astronomy but nevertheless i was at first a little sceptical about the plot of this movie it follows three children that were all born during a solar eclipse and so have no emotion and thus naturally become ruthless serial killers the plot does sound ridiculous at first but once you realise that a solar eclipse blo

<a name='e7'></a>
#### Exercise 7: Questions about the tokenizer

Answer the following questions:
- (1p) What is the size of the vocabulary?
- (2p) What are the special tokens apart from `[CLS]` and `[SEP]`? What are their functions?

--- YOUR ANSWERS HERE
1) The vocabulary size is 28996 tokens.
2) PAD: used to pad sequences to the same length within a batch.
   UNK: represents unknown words that are not in the vocabulary.
   MASK: used during pretraining for the masked language modeling objective.
Although UNK and MASK didn't appear in our tokenized example they are part of the tokenizer vocabulary and are used in specific situations (unknown words and masked language modeling during pretraining).

### 4.2 Loading the Model

In this section, we will load and examine the model. We will start with selecting the device we will place the model on. This will be a GPU (if one is available) or a CPU.

Google Colab offers free access to GPU, provided there is availability (also based on quotas which may vary based on your usage and the overall demand on Colab's resources). If you are working locally, then if you don't have a GPU, CPU will be selected. For the first parts of the assignment running on CPU might be okay but when we have to process the dataset a GPU will be necessary.

The following cell will select the device for us.

In [25]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cpu


Now, let's load the model from huggingface and move it (slowly because it's heavy due to the large number of parameters) on the device from the previous cell (the methods `to()`).

In [26]:
model = transformers.AutoModel.from_pretrained(model_name)
print('loaded on device:', model.device)
model.to(device)
print('moved to device', model.device)
print(model)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded on device: cpu
moved to device cpu
BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (

When loading the model you might have seen the warning about some unexpected weights. This means that the model on huggingface has some additional weights that were downloaded, but our model does not use them. In essence, you can load the same weights (as linked by our `model_name`) to load to different but related models. In our case those would be `BertForMaskedLM` or `BertForNextSentencePrediction` instead of our `BertModel`, which is loaded automatically as the `AutoModel`. Below is a way to load the weights into a different model.

In [30]:
# transformers.BertForMaskedLM.from_pretrained(model_name)

Next, let's use BERT model for inference. We will tokenize the first sentence of our dataset and pass it to the model. We set `output_hidden_states` to `True` in order to have access to the hidden states of the model. Those represent the latent representations after embedding and transformer layers.

In [27]:
tokenized = tokenizer(dataset['test'][0]['text'], padding=True, truncation=True, return_tensors='pt').to(device)
print(tokenized)
model_output = model(**tokenized, output_hidden_states=True)

{'input_ids': tensor([[  101,  1142,  4094,  3189,  2515,  1185,   188,  5674, 25614,  1116,
          1290,  1103,  2523,   188,  5674,  8825,  2111,  1122,  1110,  4122,
          1105,   185, 14604, 14964,  1665,  1122,  1144,  1185,  7740, 11116,
         20748,  1191,  1800, 20384,  1128,  1106,  1117,  1402,  1106,  2824,
           182,  1197, 24034,  9866,  1818,  1274,  1204,  1301,  1256,  1103,
          1641,  1104,  1103,  2523,  1110, 25671,  1105,  4104, 22367,  1184,
         14946,  2887,  1439,  1103,  2006,  4928,  1110,  1177,  2213,  1115,
          1122,  1180,  2797,  1534,  1175,  3202,  1106, 15546,  1122,  1445,
          1204,   170,  6625,  1115,  1521,  1106,  1103,  1346, 14353,  1104,
          2869,   174, 13409,  1122,  1108,  1515,  1106,  2496,  1107,  1142,
           172,  7535,  6773,  1200,  1115,  1225,  1140,  1107,  1122,  1538,
          1138, 13526,  1140,  1103,  1832,  1104,  1117,  1552,  3229,  1119,
          1108,  1126,  3437,  8143,  

Examine the next cell and make sure everything makes sense to you. Consult the [documentation](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertModel.forward) in case of doubt.

In [28]:
print(list(model_output.keys()))
print()
print('pooler_output:')
print(type(model_output['pooler_output']))
print(model_output['pooler_output'].shape)
print()
print('hidden_states:')
print(type(model_output['hidden_states']))
print(len(model_output['hidden_states']))
print(type(model_output['hidden_states'][0]))
print(model_output['hidden_states'][0].shape)
print()
print('last_hidden_state:')
print(type(model_output['last_hidden_state']))
print(model_output['last_hidden_state'].shape)
print(model.num_parameters())
print(model.embeddings.word_embeddings.num_embeddings)

['last_hidden_state', 'pooler_output', 'hidden_states']

pooler_output:
<class 'torch.Tensor'>
torch.Size([1, 768])

hidden_states:
<class 'tuple'>
13
<class 'torch.Tensor'>
torch.Size([1, 149, 768])

last_hidden_state:
<class 'torch.Tensor'>
torch.Size([1, 149, 768])
108310272
28996


<a name='e8'></a>
#### Exercise 8: Questions about the Model

Examine the output of the previous cells. Answer the following questions:
- (1p) What is the number of transformer layers in this model?
- (1p) What is the dimension of the embeddings?
- (1p) What is the hidden size of the FFN in the transformer layer?
- (1p) What is the total number of parameters of the model (hint: check the `num_parameters()` method of the model)?
- (1p) How can you find the vocabulary size from the model?
- (1p) What is the length of the `hidden_states` in the output? Why?

--- YOUR ANSWERS HERE
1) The number of transformer layers is 12
2) Dimensionality of each embedding is 768
3) Hidden size of the FFN in the transformer layer is 3072
4) Total number of parameters is 108310272
5) Vocabulary size can be found through the number of embeddings - 28996
6) Length of the hidden_states in the output is 13 as there are 12 transformer layers in BERT model + output from one embedding block (Hidden-states of the model at the output of each layer plus the optional initial embedding outputs.)

## 5. BERT Sentence Embeddings

Having the model loaded and ready we can work on obtaining the sentence embeddings. During the last lab, you averaged the token embeddings. This time we will start with something else. Remember the CLS token? Its hidden representation is often used for classification as a representation of the whole sentence. We will do exactly that.

But first, we have to tokenize the dataset using BERT tokenizer.

<a name='e9'></a>
#### Exercise 9: BERT tokenizing examples

(5p) Fill in the following function to embed the examples (passed as a parameter) using the tokenizer (also a parameter). The function will tokenize a batch of examples, but the tokenizer can handle that, if you remember from the previous section.

In [29]:
def tokenize_text_bert(examples, tokenizer):
    """
    Tokenizes the `sentence` column from the batch of examples and returns the whole output of the tokenizer.
    Args:
        examples: a batch of examples
        tokenizer: the BERT tokenizer

    Returns: the tokenized `sentence` column (returns the whole output of the tokenizer)

    """
    ### YOUR CODE HERE
    tokenized_sentence = tokenizer(
        examples["text"], padding = True, truncation = True, max_length = 512)
    
    ### YOUR CODE ENDS HERE
    return tokenized_sentence


In [30]:
dataset_tokenized_bert = dataset.map(tokenize_text_bert,
                                     fn_kwargs={'tokenizer': tokenizer},
                                     batched=True,
                                     remove_columns=dataset['train'].column_names,)
print(dataset_tokenized_bert)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
})


<a name='e10'></a>
#### Exercise 10: BERT sentence embeddings by the CLS token

(5p) Implement the following function which calculates the sentence embeddings based on the model output (passed to the function as a parameter). It should take the embedding of the CLS token of last layer.

In [31]:
def calculate_cls_embeddings(input_batch, model_output):
    """
    Calculates the sentence embeddings of a batch of sentences as the last-layer representation of the CLS token.
    Args:
        input_batch: tokenized batch of sentences (as returned by the tokenizer), contains `input_ids`, `token_type_ids`, and `attention_mask` tensors
        model_output: the output of the model given the `input_batch`, contains `last_hidden_state`, `pooler_output`, `hidden_states` tensors

    Returns: tensor of the hidden states of the CLS token (from the last layer) for each example in the batch

    """

    ### YOUR CODE HERE
    last_hidden_state = model_output['last_hidden_state']
    sentence_embeddings = last_hidden_state[:, 0, :]

    ### YOUR CODE ENDS HERE

    return sentence_embeddings

In [32]:
text = "The weather is nice today."
tokenized = tokenizer(text, padding=True, return_tensors='pt').to(device)
print(tokenized)
model_output = model(**tokenized, output_hidden_states=True)
print(model_output['last_hidden_state'].shape)
sentence_embedding = calculate_cls_embeddings(tokenized, model_output)
print(sentence_embedding.shape)

{'input_ids': tensor([[ 101, 1109, 4250, 1110, 3505, 2052,  119,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 8, 768])
torch.Size([1, 768])


In [33]:
def embed_dataset(dataset, model, sentence_embedding_fn, batch_size=8):
    data_collator = transformers.DataCollatorWithPadding(tokenizer)
    data_loader = DataLoader(dataset, batch_size=batch_size, collate_fn=data_collator)
    sentence_embeddings = []
    with torch.no_grad():
        for batch in tqdm.tqdm(data_loader):
            batch.to(device)
            model_output = model(**batch, output_hidden_states=True)
            batch_sentence_embeddings = sentence_embedding_fn(batch, model_output)
            sentence_embeddings.append(batch_sentence_embeddings.detach().cpu())

    sentence_embeddings = torch.concat(sentence_embeddings, dim=0)
    return sentence_embeddings

In [34]:
bert_cls_train = embed_dataset(dataset_tokenized_bert['train'], model, calculate_cls_embeddings)
print(bert_cls_train.shape)

bert_cls_valid = embed_dataset(dataset_tokenized_bert['validation'], model, calculate_cls_embeddings)
print(bert_cls_valid.shape)

100%|███████████████████████████████████████████████████████████| 2500/2500 [27:31<00:00,  1.51it/s]


torch.Size([20000, 768])


100%|█████████████████████████████████████████████████████████████| 625/625 [06:45<00:00,  1.54it/s]

torch.Size([5000, 768])


In [37]:
classifier = LogisticRegression(solver='lbfgs', max_iter=1000)
print('Training classifier...')
classifier.fit(bert_cls_train, y_train)

Training classifier...


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [38]:
print('BERT train')
pred_train = classifier.predict(bert_cls_train)
print(accuracy_score(y_train, pred_train))

print('BERT valid')
pred_valid = classifier.predict(bert_cls_valid)
print(accuracy_score(y_valid, pred_valid))

BERT train
0.85215
BERT valid
0.8374


You can test the model on the evaluation metric of your choice:

In [39]:
#### YOUR CODE HERE
print("BERT validation F1-score:")
pred_valid = classifier.predict(bert_cls_valid)
print(f1_score(y_valid, pred_valid))
print("Precision:")
print(precision_score(y_valid, pred_valid))

#### YOUR CODE ENDS HERE

BERT validation F1-score:
0.8353917797124925
Precision:
0.8416972664218686


<a name='e11'></a>
#### Exercise 11: BERT Sentence embeddings by averaging tokens

(5p) Implement embedding sentences by averaging the hidden representations of tokens. Make sure to ignore the special and padding tokens. The padding tokens are indicated by the attention mask. You can find the other special tokens using the tokenizer's attributes such as `tokenizer.sep_token_id`. The function accepts the `layer` parameter. Typically, you would use the hidden representations of the last layer, it might be beneficial for some tasks to use previous layers or an averaged representations of multiple layers.

In [40]:
### YOUR CODE HERE

def calculate_sentence_embeddings(input_batch, model_output, layer=-1):
    """
    Calculates the sentence embeddings of a batch of sentences as a mean of token representations.
    The representations are taken from the layer of the index provided as a `layer` parameter.
    Args:
        input_batch: tokenized batch of sentences (as returned by the tokenizer), contains `input_ids`, `token_type_ids`, and `attention_mask` tensors
        model_output: the output of the model given the `input_batch`, contains `last_hidden_state`, `pooler_output`, `hidden_states` tensors
        layer: specifies the layer of the hidden states that are used to calculate sentence embedding

    Returns: tensor of the averaged hidden states (from the specified layer) for each example in the batch

    """
    attention_mask = input_batch['attention_mask']
    hidden_states = model_output['hidden_states'][layer]

    ### YOUR CODE HERE
    mask = attention_mask.unsqueeze(-1)
    masked_hidden_states = hidden_states * mask
    sum_hidden = masked_hidden_states.sum(dim=1)
    lengths = mask.sum(dim=1)
    sentence_embeddings = sum_hidden / lengths




    ### YOUR CODE ENDS HERE

    return sentence_embeddings

### YOUR CODE ENDS HERE

We can test it here:


In [41]:
text = "The weather is nice today."
tokenized = tokenizer(text, padding=True, return_tensors='pt').to(device)
print(tokenized)
model_output = model(**tokenized, output_hidden_states=True)
print(model_output['last_hidden_state'].shape)
sentence_embedding = calculate_sentence_embeddings(tokenized, model_output)
print(sentence_embedding.shape)

{'input_ids': tensor([[ 101, 1109, 4250, 1110, 3505, 2052,  119,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 8, 768])
torch.Size([1, 768])


We will embed the sentences and evaluate the model on the `validation` subset.


In [42]:
bert_sentence_train = embed_dataset(dataset_tokenized_bert['train'], model, calculate_sentence_embeddings)
print(bert_cls_train.shape)

bert_sentence_valid = embed_dataset(dataset_tokenized_bert['validation'], model, calculate_sentence_embeddings)
print(bert_cls_valid.shape)

100%|███████████████████████████████████████████████████████████| 2500/2500 [32:26<00:00,  1.28it/s]


torch.Size([20000, 768])


100%|█████████████████████████████████████████████████████████████| 625/625 [08:18<00:00,  1.25it/s]

torch.Size([5000, 768])


In [43]:
classifier = LogisticRegression(solver='lbfgs', max_iter=1000)
print('Training classifier...')
classifier.fit(bert_sentence_train, y_train)

Training classifier...


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [44]:
print('BERT train')
pred_train = classifier.predict(bert_sentence_train)
print(accuracy_score(y_train, pred_train))

print('BERT valid')
pred_valid = classifier.predict(bert_sentence_valid)
print(accuracy_score(y_valid, pred_valid))

BERT train
0.8866
BERT valid
0.8732


Test the model on the evaluation metric of your choice:


In [45]:
#### YOUR CODE HERE
pred_valid = classifier.predict(bert_cls_valid)
print("BERT validation F1-score:")
print(f1_score(y_valid, pred_valid))

#### YOUR CODE ENDS HERE

BERT validation F1-score:
0.7212537418559606


## 6. Testing all methods

In this last section, you will bering together all of what you have done so far in this lab. First, you will find the best classifier. Next, you will evaluate all the models you created so far.

<a name='e12'></a>
#### Exercise 12: Find the best classifier for the models

(5p) Basically, do what the title of the exercise says. Evaluate on the `validation` subset. Try at least two other classifiers (apart from the logistic regression). Comment on the results.

In [46]:
#### YOUR CODE HERE
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
X_train = bert_sentence_train
X_valid = bert_sentence_valid

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM (LinearSVC)": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42),
}

print("Validation results:")
for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred_valid = clf.predict(X_valid)

    acc = accuracy_score(y_valid, pred_valid)
    f1 = f1_score(y_valid, pred_valid)

    print(f"{name}: accuracy={acc:.4f}, f1={f1:.4f}")

#### YOUR CODE ENDS HERE

Validation results:
Logistic Regression: accuracy=0.8732, f1=0.8718
Linear SVM (LinearSVC): accuracy=0.8728, f1=0.8718
Random Forest: accuracy=0.8124, f1=0.8083


--- YOUR ANSWERS HERE
The results show that Logistic Regression works best for this task, linear SVM performs very similarly to Logistic Regression, but slightly worse. This indicates that the BERT sentence embeddings can be separated quite well using simple linear decision boundaries
Random Forest gives lower accuracy and F1-score compared to the linear models. This shows that tree-based models are less suitable for high-dimensional sentence embeddings
Overall results show that simple linear classifiers are sufficient and more effective when using BERT sentence embeddings and adding more complex models does not improve performance.

<a name='e13'></a>
#### Exercise 13: Evaluating methods on the test set

(10p) Test the models you implemented on the test subset:
- Hand-crafted features,
- BOW,
- BERT model based on the CLS token.

You have the models trained already, so only do evaluation.

Evaluate the performance using the metric(s) of your choice. Make sure to discuss the results. Which model performed best? Is this what you expected?

In [47]:
#### YOUR CODE HERE


y_test = dataset["test"]["label"]

def evaluate_and_print(name, clf, Xtr, ytr, Xte, yte):
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    print(f"\n{name}")
    print(f"  accuracy : {accuracy_score(yte, pred):.4f}")
    print(f"  precision: {precision_score(yte, pred):.4f}")
    print(f"  recall   : {recall_score(yte, pred):.4f}")
    print(f"  f1       : {f1_score(yte, pred):.4f}")
    return pred

print("=== TEST SET EVALUATION (Exercise 13) ===")


X_train_feat = calculate_features_dataset(dataset["train"], calculate_features)
X_test_feat  = calculate_features_dataset(dataset["test"],  calculate_features)

clf_feat = LogisticRegression(solver="lbfgs", max_iter=1000)
_ = evaluate_and_print("Hand-crafted features (LogReg)", clf_feat, X_train_feat, y_train, X_test_feat, y_test)

test_bows = []
for ex in tqdm.tqdm(dataset["test"], desc="Calculating test BOWs"):
    test_bows.append(bag_of_words(ex["tokens"], token_to_id))
test_bows = np.array(test_bows, dtype=float)

clf_bow = LogisticRegression(solver="lbfgs", max_iter=1000)
_ = evaluate_and_print("BOW (LogReg)", clf_bow, train_bows, y_train, test_bows, y_test)


bert_model = transformers.AutoModel.from_pretrained(model_name).to(device)
bert_model.eval()
bert_cls_test = embed_dataset(dataset_tokenized_bert["test"], bert_model, calculate_cls_embeddings)
clf_bert = LogisticRegression(solver="lbfgs", max_iter=1000)
_ = evaluate_and_print("BERT CLS (LogReg)", clf_bert, bert_cls_train, y_train, bert_cls_test, y_test)

### YOUR CODE ENDS HERE

=== TEST SET EVALUATION (Exercise 13) ===


Extracting features: 100%|██████████████████████████████████| 25000/25000 [00:03<00:00, 7114.76it/s]



Hand-crafted features (LogReg)
  accuracy : 0.7151
  precision: 0.6738
  recall   : 0.8340
  f1       : 0.7454


Calculating test BOWs: 100%|████████████████████████████████| 25000/25000 [00:02<00:00, 8348.63it/s]



BOW (LogReg)
  accuracy : 0.8594
  precision: 0.8506
  recall   : 0.8718
  f1       : 0.8611


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|███████████████████████████████████████████████████████████| 3125/3125 [34:57<00:00,  1.49it/s]



BERT CLS (LogReg)
  accuracy : 0.8392
  precision: 0.8425
  recall   : 0.8343
  f1       : 0.8384


--- YOUR ANSWERS HERE
The model with hand-crafted features achieved the lowest performance. This is expected because these features describe the text only in a very simple way and dont take into account which words are actually used.
The Bag-of-Words model achieved the best results, because for sentiment analysis the presence of specific words is very important, and Bag-of-Words represents this information well.
The BERT CLS model performed slightly worse than Bag-of-Words, because BERT was used only as a fixed feature extractor and wasnt fine-tuned for this task. In addition, the CLS embedding isnt always the best representation for text classification.
Overall, the results match our expectations. Even though BERT is a more advanced model, without fine-tuning it doesnt necessarily outperform a strong Bag-of-Words baseline.